# EverFlavor AI - Computer Vision: Food Freshness From Photos

**Team:** EverGlow
**Course:** ITAI 2277
**Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

> **Status: planning sketch.** The sections, sources, output tables and functions below are the plan.
> The functions are stubs (`NotImplementedError`) and nothing here has been run or tested yet.
> "Run all" is safe: it only runs the setup and prints the plan's status.

---

## Purpose

The project's **computer vision** pipeline (image classification). From a photo of an ingredient, say
how fresh it looks (fresh, getting old, past its best) and, where the data allows, roughly how long it has
been stored. That is combined with USDA shelf-life guidance from notebook 04.

It covers four food groups, because freshness looks different in each:

| Group | What the camera sees |
|---|---|
| Fruits and vegetables | Color, wrinkles, soft spots, mold |
| Red meat | Color (bright red -> brown / grey), surface dryness or slime |
| Fish | Eyes (clear and bulging -> cloudy and sunken), gills (red -> brown) |
| Bread | Mold spots |

This is a different kind of pipeline from notebooks 01-04: the data is **images**, so it needs its own
collection, labeling, splitting and model training, usually with a GPU (Colab: Runtime > Change runtime
type > GPU).

**Safety rule:** the model only ever says how food **looks**. It never says food is safe to eat: spoilage
and harmful bacteria are often invisible, above all in meat and fish. Every answer ends with "check the
smell, texture and date; when in doubt, throw it out" (the USDA advice), and for meat and fish it adds
the USDA storage time from notebook 04.

## Inputs and outputs

| | File | Made by |
|---|---|---|
| Input | Public image datasets (section 2; licenses checked first) | Section 2 |
| Input | `data/processed/ingredient_shelf_life.parquet` | Notebook 04 |
| Output | `data/processed/freshness_images.parquet` (one row per image, section 3) | Sections 2-4 |
| Output | `data/processed/freshness_coverage.csv` (images per group, item, state and source) | Section 4 |
| Output | `models/freshness/` (the trained models, not stored in GitHub) | Section 5 |

## Contents

1. Setup
2. Image Sources and Licenses
3. One Label Scheme for Every Source
4. Cleaning, Coverage and Splitting Without Leakage
5. Model Plan
6. Team Decisions and Next Steps

---

## 1. Setup

The same setup as notebooks 01 and 02: it works unchanged in Google Colab, VS Code and Antigravity.

In [ ]:
# %pip (not !pip) installs into the Python that runs this notebook,
# in Colab and in local kernels (VS Code / Antigravity / JupyterLab).
%pip install -q requests pandas pyarrow pillow imagehash

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# -------------------------------------------------------
# 1. Where is this notebook running? Move to the project folder
#    (the one that holds notebooks/ and data/), as in notebook 01.
# -------------------------------------------------------
try:
    import google.colab  # noqa: F401  # pyright: ignore[reportMissingImports]  (only exists in Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_project_root(start):
    # Return the nearest folder at or above `start` with notebooks/ and data/, or None
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# -------------------------------------------------------
# 2. The project's code (src/everflavor). If the notebook was opened on its
#    own, the missing files are downloaded from GitHub (temporary name
#    first, __init__.py last), exactly as in notebook 01.
# -------------------------------------------------------
SRC_DIR = PROJECT_ROOT / "src"
EVERFLAVOR_MODULES = ["__init__", "charts", "checks", "cooking", "cuisine", "diets", "environment",
                      "features", "flags", "ingredients", "knowledge", "nutrition", "parsing", "pipeline",
                      "progress", "recommend", "reporting", "review", "sources"]
EVERFLAVOR_URL = "https://raw.githubusercontent.com/Chloe-Tu2/The-EverFlavor-AI/main/src/everflavor/"
EVERFLAVOR_DIR = SRC_DIR / "everflavor"
missing_modules = [m for m in EVERFLAVOR_MODULES if not (EVERFLAVOR_DIR / f"{m}.py").exists()]
if missing_modules:
    import urllib.request
    EVERFLAVOR_DIR.mkdir(parents=True, exist_ok=True)
    for module in sorted(missing_modules, key=lambda m: m == "__init__"):
        partial = EVERFLAVOR_DIR / f"{module}.py.part"
        urllib.request.urlretrieve(f"{EVERFLAVOR_URL}{module}.py", partial)
        partial.replace(EVERFLAVOR_DIR / f"{module}.py")
    print(f"Downloaded {len(missing_modules)} project code file(s) (src/everflavor) from GitHub.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from everflavor.environment import short_path

REFRESH_DOWNLOADS = False   # True downloads every source again instead of using the saved copy
SPLIT_FILES = {s: Path(f"data/processed/recipes_{s}.parquet") for s in ["train", "val", "test"]}
ALL_RECIPES_FILE = Path("data/interim/recipes_all.parquet")
IMAGE_DIR = Path("data/raw/freshness_images")   # images are never committed (too large)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Runtime        : {'Google Colab' if IN_COLAB else 'Local Jupyter (VS Code / Antigravity / JupyterLab)'}")
print(f"Python         : {sys.version.split()[0]}")
print(f"Project folder : {short_path(PROJECT_ROOT)}")
missing_inputs = [str(p) for p in [*SPLIT_FILES.values(), ALL_RECIPES_FILE] if not p.exists()]
if missing_inputs:
    print("\nMISSING notebook 01 outputs:", ", ".join(missing_inputs))
    print("Run notebook 01 first (all of Week 5), then run this notebook again.")
else:
    print("Notebook 01 outputs found.")

---

## 2. Image Sources and Licenses

**Team choice: use many sources, even when they overlap.** Photos of the same food taken by different
people, in different countries, years, seasons, kitchens and markets, with different cameras, light and
angles, teach the model what freshness looks like instead of what one lab's table looks like. Overlap is
kept on purpose; only true copies of the same photo are removed (section 4), and each source is weighted
so a large dataset does not drown out a small one (section 5).
Licenses were read on 2026-10-04. **CC BY** means free to use with credit to the authors; **CC BY-SA**
also requires sharing changed versions under the same license. "To confirm" means the license was read
on the article, and the data record itself must be checked before download.

### Fruits and vegetables

| Dataset | What it has | License | What it adds |
|---|---|---|---|
| **AgriFreshNET** (Mendeley 42m5tb7yv9) | 14,500 phone photos, 24 fruits and vegetables, **fresh / semi-fresh / rotten**, with an approximate shelf life per sample (for example 1-4 or 8-12 days); markets, homes, farms | CC BY 4.0 | **Main source:** the middle state, real-world photos, shelf-life ranges |
| **TriModal Ripeness 6** (Figshare 30783827) | Guava, carrot, tomato, Indian gooseberry, banana, mango photographed **3 times a day until they spoil**, plus thermal images and methane readings | CC BY | **Storage time:** exact hours since the start. 19 GB, so only the color photos are downloaded |
| **FruitNet** (Mendeley b6fftwbr2v) | 14,700 photos of 6 Indian fruits, good / bad quality, different backgrounds and light | CC BY 4.0 | More examples of 6 common fruits |
| **BananaImageBD** (Mendeley ptfscwtnyz) | Bananas in 4 stages: green, semi-ripe, ripe, overripe (820 original photos) | CC BY 4.0 | Ripeness stages for the most common fruit |
| Fruits-360 (Kaggle) | 250+ classes, studio photos on white | CC BY-SA 4.0 | Item recognition only (which fruit is it), not freshness |

### Red meat

| Dataset | What it has | License | What it adds |
|---|---|---|---|
| **MeatScan** (Zenodo 10.5281/zenodo.16764338) | 6,995 original photos of beef, fresh vs spoiled, in markets, butcher shops and cold storage; 3 labelers agreed (Fleiss' kappa 0.87) | CC BY (to confirm) | Real market photos with checked labels |
| Meat freshness (Roboflow Universe) | About 2,000 photos, fresh / half-fresh / spoiled | CC BY 4.0 (to confirm) | The middle state for meat |

### Fish

| Dataset | What it has | License | What it adds |
|---|---|---|---|
| **DaFiF** (Mendeley vx4ptwk3pb) | Mackerel, tilapia, tuna photographed daily for **11 days**, with an expert freshness score (1-9) for eyes, gills, skin, flesh, smell and texture; also gas sensor readings | CC BY (to confirm) | **Storage time and expert scores**, not just fresh / spoiled |
| **Freshness of the Fish Eyes** (Mendeley xzyx7pbr3w) | 4,392 eye photos, 8 species, days 1-2 / 3-4 / 5-6 | CC BY 4.0 | Many species; day ranges |
| **Multistage Fish Eyes** (Mendeley 67nmx3mhwh) | 4,800 eye photos, 2 freshwater species, 4 light and background setups, days ≤2 / 2-4 / >4 | CC BY 4.0 | Balanced classes, varied lighting |

### Bread

| Dataset | What it has | License | What it adds |
|---|---|---|---|
| Bread mold detection (Roboflow Universe) | About 900 photos with mold marked by boxes | CC BY 4.0 (to confirm) | Mold, the main sign for baked goods |

### Our own photos

| Dataset | What it has | License | What it adds |
|---|---|---|---|
| Team photos | Groceries bought locally, photographed every day in a real kitchen | Team consent | **The final test:** most public photos were taken in India, Bangladesh, Pakistan, Ghana, Indonesia and China, so this checks the model on local produce, cameras and kitchens |

**Not used:** datasets with no stated license, and augmented copies inside a dataset (MeatScan and
BananaImageBD ship rotated and flipped copies). Only original photos are kept, so copies of one photo
cannot end up in both training and test.

In [ ]:
# Planned module: src/everflavor/freshness.py
IMAGE_SOURCES = pd.DataFrame([
    # source, food group, host, license, labels, storage time known
    ("agrifreshnet", "produce", "mendeley 42m5tb7yv9", "CC BY 4.0", "fresh/semi-fresh/rotten", "shelf-life range"),
    ("trimodal_ripeness6", "produce", "figshare 30783827", "CC BY", "time series", "hours"),
    ("fruitnet", "produce", "mendeley b6fftwbr2v", "CC BY 4.0", "good/bad", "no"),
    ("bananaimagebd", "produce", "mendeley ptfscwtnyz", "CC BY 4.0", "green/semi-ripe/ripe/overripe", "no"),
    ("meatscan", "meat", "zenodo 16764338", "CC BY (to confirm)", "fresh/spoiled", "no"),
    ("roboflow_meat", "meat", "roboflow universe", "CC BY 4.0 (to confirm)", "fresh/half-fresh/spoiled", "no"),
    ("dafif", "fish", "mendeley vx4ptwk3pb", "CC BY (to confirm)", "expert score 1-9", "days"),
    ("fish_eyes_ffe", "fish", "mendeley xzyx7pbr3w", "CC BY 4.0", "days 1-2/3-4/5-6", "day range"),
    ("fish_eyes_mfed", "fish", "mendeley 67nmx3mhwh", "CC BY 4.0", "days <=2/2-4/>4", "day range"),
    ("roboflow_bread_mold", "bread", "roboflow universe", "CC BY 4.0 (to confirm)", "mold boxes", "no"),
    ("team_photos", "all", "team", "team consent", "day by day", "days"),
], columns=["source", "group", "host", "license", "labels", "storage_time"])
print(IMAGE_SOURCES.to_string(index=False))

---

## 3. One Label Scheme for Every Source

Every source labels differently, so each is mapped to one scheme (like cuisine families in notebook 01).
Two targets: the **state** (every image has one) and the **storage age** (only images whose source
records it).

| Column | Values |
|---|---|
| `group` | `produce`, `meat`, `fish`, `bread` |
| `item` | Our ingredient name (`apple`, `banana`, `beef`, `tilapia` ...) |
| `state` | `fresh`, `aging`, `spoiled` |
| `age_days_min`, `age_days_max` | Storage time when the source records it (an exact day gives min = max); empty otherwise |
| `expert_score` | DaFiF's 1-9 freshness score; empty for other sources |
| `photo_set` | One fish, one fruit or one photo session: all its photos stay in one split |
| `source`, `license` | Where the image came from |

The state is mapped from each source's own words. Sources that only give storage days (the fish eye
sets) get a state from the day ranges, and that rule is written down so the team can change it.

In [ ]:
STATE_MAP = {
    "fresh": "fresh", "good": "fresh", "highly fresh": "fresh", "green": "fresh", "semi-ripe": "fresh",
    "semi-fresh": "aging", "half-fresh": "aging", "ripe": "aging", "overripe": "aging",
    "rotten": "spoiled", "bad": "spoiled", "spoiled": "spoiled", "not fresh": "spoiled",
}
# Left out on purpose: FruitNet "mixed" (good and bad fruit in one photo).
# Team decision: is an overripe banana "aging" or "spoiled"? (section 6)


def index_images(source: str, folder: Path) -> pd.DataFrame:
    """One row per original image (augmented copies skipped) with the columns of the label scheme."""
    raise NotImplementedError("sketch: image index")

---

## 4. Cleaning, Coverage and Splitting Without Leakage

- **Duplicates:** near-identical photos (the same fruit from two angles, or a dataset that copies
  another) are found with a perceptual hash and kept in one split only, like `ingredient_group` in
  notebook 01.
- **Split by `photo_set`**, never by single image: DaFiF photographs the same fish for 11 days, so a
  random split would put day 3 of a fish in training and day 4 of the same fish in test.
- **Hold out whole sources** as an extra test: train without one source and test on it, to see whether
  the model works on photos from a place and camera it has never seen.
- **Coverage report** (`freshness_coverage.csv`): images per group, item, state and source. It shows which
  items have too few photos, the same way notebook 03 reports coverage by country.

In [ ]:
def group_near_duplicates(index: pd.DataFrame, max_distance: int = 4) -> pd.Series:
    """Group images whose perceptual hashes (imagehash.phash) differ by at most `max_distance` bits."""
    raise NotImplementedError("sketch: duplicate groups")


def freshness_coverage(index: pd.DataFrame) -> pd.DataFrame:
    """Images per group, item, state and source, with a warning column for items under a minimum count."""
    raise NotImplementedError("sketch: coverage report")

---

## 5. Model and Training Plan

1. **One model per food group** to start (produce, meat, fish, bread): a fish eye and a strawberry share
   little, and a small model per group is easier to check.
2. Fine-tune a small pretrained image model (for example MobileNet or EfficientNet from `torchvision` or
   `timm`) on `state`.
3. Where storage age is known, a second output predicts the age range ("about 3-4 days"). It is trained
   only on images that have an age.
4. **Report:** accuracy per item, a confusion matrix, and above all how often **spoiled food is called
   fresh** (the costly error). The decision threshold is set to keep that error low, even if more fresh
   food gets a "check it" warning.
5. **Test** on held-out sources and on the team's own photos before any agent uses the model.

### How many epochs?

No fixed number: the **validation split decides when to stop**.

| Phase | What trains | Epochs | Learning rate |
|---|---|---|---|
| 1. Head only | Only the new last layer; the pretrained model is frozen | About 5 | Normal (for example 1e-3) |
| 2. Fine-tuning | The whole model | Up to about 30, with **early stopping** | Lower (for example 1e-4), reduced when validation stops improving |

- **Early stopping:** after every epoch the model is scored on the validation split. If the score has not
  improved for 5 epochs (`PATIENCE`), training stops and the **best** epoch is kept, not the last one.
  Fine-tuning on datasets of this size usually settles within 10-20 epochs.
- **The score that picks the best epoch** puts safety first: the share of spoiled photos called fresh,
  then macro F1 (accuracy that counts every state equally).
- **Balance by source:** each photo is weighted so every source counts about equally in training, and
  within a source every state counts equally.
- **Augmentation during training** (random crops, flips, small color and brightness changes) helps the
  model cope with other cameras and light. It is applied on the fly, never saved as extra files.
- **The test split is used once, at the very end.** Checking it after every change would tune the model
  to it, and the final number would look better than the model really is.
- **Repeat with 3 random seeds** and report the average and spread, so one lucky run is not mistaken for
  a good model.

In [ ]:
TRAINING_PLAN = {
    "head_only_epochs": 5,
    "max_finetune_epochs": 30,
    "patience": 5,               # epochs without improvement before stopping
    "head_learning_rate": 1e-3,
    "finetune_learning_rate": 1e-4,
    "selection_metric": "spoiled_called_fresh (lower is better), then macro F1",
    "weights": "equal per source, then equal per state within a source",
    "seeds": [0, 1, 2],
}


def train_freshness_model(index: pd.DataFrame, group: str, plan: dict = TRAINING_PLAN) -> dict:
    """Train one food group's model with early stopping on the validation split; never reads the test split.

    Returns the best epoch, its validation scores and the saved model path.
    """
    raise NotImplementedError("sketch: training loop")

---

## 6. Team Decisions and Next Steps

1. **Pilot items:** a short list per group, picked from the coverage report.
2. **State rules:** is an overripe banana "aging" or "spoiled"? How many storage days make fish "aging"?
3. **Team photos:** who takes them, how often, and a fixed routine (same light, same angle, date in the
   file name).
4. **Licenses marked "to confirm":** check the data record before download.

**Build order:** produce first (AgriFreshNET, FruitNet), then fish (the three Mendeley sets), then meat,
then bread.

---

## Plan status

What is built, what is a stub, and what needs a team decision first.

In [ ]:
PLAN = pd.DataFrame([
    ("Plan, sources, label scheme", "written"),
    ("2 Image sources (11, 4 food groups)", "licenses read; 4 to confirm on the data record"),
    ("3 Image index", "stub"),
    ("4 Duplicates, coverage and split", "stub"),
    ("5 Models (one per food group)", "plan only"),
], columns=["step", "status"])
print(PLAN.to_string(index=False))